# EV-Drone-Detector — Colab (Train + Detect)

SPGNet tabanlı event-camera drone init-detection pipeline'ı. Sırayla çalıştır:
1. GPU / CUDA doğrula
2. Repo'yu klonla
3. Bağımlılıkları kur (`spconv-cu118`)
4. Smoke test (synthetic) — spconv çalışıyor mu?
5. EV-UAV dataset'ini Drive'dan bağla
6. Eğitim
7. Detection + görselleştirme

**Runtime → Change runtime type → GPU (T4/A100).**

## 1. GPU doğrulama

In [ ]:
!nvidia-smi | head -20
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Repo'yu klonla
Private repo ise `https://<TOKEN>@github.com/...` formatı kullan.

In [ ]:
REPO_URL = 'https://github.com/yigitkayabagci/ev-drone-detector.git'
REPO_DIR = '/content/ev-drone-detector'

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull --ff-only || true
!ls

## 3. Bağımlılıkları kur (Colab CUDA 11.8 → spconv-cu118)
İlk kurulumdan sonra **Runtime → Restart session** yapıp buradan devam et.

In [ ]:
%cd /content/ev-drone-detector
!pip install -q -e '.[colab,dev]'
print('\nDone. If spconv was just installed, restart the runtime (Runtime → Restart session) before running further cells.')

In [ ]:
%cd /content/ev-drone-detector
import spconv, torch
print('spconv:', spconv.__version__, '| torch:', torch.__version__, '| cuda:', torch.version.cuda)

## 4. Smoke test — tüm testler (spconv dahil) pass olmalı

In [ ]:
%cd /content/ev-drone-detector
!python -m pytest tests/ -v

## 5. Synthetic veriyle 1-epoch duman eğitimi
Pipeline uçtan uca çalışıyor mu görmek için.

In [ ]:
%cd /content/ev-drone-detector
!python scripts/train.py --config configs/default.yaml --synthetic 2>&1 | tail -40

> `configs/default.yaml`'daki `epochs: 50` synthetic smoke test için fazla olabilir. Hızlı deneme istersen yaml'ı geçici düşür veya ayrı bir config dosyası yap.

## 6. EV-UAV dataset'ini Drive'dan bağla
Dataset'i önceden Google Drive'a yükle: `MyDrive/ev_uav/{train,val,test}/*.npz`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_EV_UAV = '/content/drive/MyDrive/ev_uav'  # Drive'daki yol
REPO_DATA = '/content/ev-drone-detector/data'

!mkdir -p {REPO_DATA}
!ln -sfn {DRIVE_EV_UAV}/train {REPO_DATA}/train
!ln -sfn {DRIVE_EV_UAV}/val   {REPO_DATA}/val
!ln -sfn {DRIVE_EV_UAV}/test  {REPO_DATA}/test
!ls -la {REPO_DATA}

In [ ]:
# Tek .npz sanity check: evs_norm (N,5=x,y,t,p,label) + ev_loc (N,3) bekleniyor
import glob, numpy as np
samples = glob.glob('/content/ev-drone-detector/data/train/*.npz')
assert samples, 'train/ boş — Drive yolunu kontrol et'
d = np.load(samples[0])
print('keys :', list(d.keys()))
print('evs_norm:', d['evs_norm'].shape, d['evs_norm'].dtype)
print('ev_loc  :', d['ev_loc'].shape, d['ev_loc'].dtype)

## 7. Gerçek eğitim
Default config: Adam lr=1e-3, StepLR(10, 0.1), 50 epoch, bs=1. `checkpoints/best_iou.pt` üretilecek.

Uzun sürer; Colab oturumunu açık tut. Drive'a checkpoint yazmak istersen `cfg.training.checkpoint_dir`'i Drive altına yönlendir.

In [ ]:
%cd /content/ev-drone-detector
!python scripts/train.py --config configs/default.yaml --device cuda:0

## 8. Detection + görselleştirme
Tek dosya veya bir klasör üzerinde çalıştır. `--visualize` ile bbox'lı PNG'ler üretilir.

In [ ]:
%cd /content/ev-drone-detector
!python scripts/detect.py \
    --config configs/default.yaml \
    --checkpoint checkpoints/best_iou.pt \
    --input data/test/ \
    --output detections.json \
    --visualize --vis_dir visualizations \
    --device cuda:0
!ls visualizations | head

In [ ]:
# İlk birkaç görselleştirmeyi inline göster
import glob
from IPython.display import Image, display
for p in sorted(glob.glob('/content/ev-drone-detector/visualizations/*.png'))[:5]:
    print(p)
    display(Image(p))

## 9. (Ops.) Programatik kullanım — tracker'a init box besleme
Her `.npz` için dönen `bbox = [x_min, y_min, x_max, y_max]` + `score`. En yüksek skorlu kutu downstream tracker'ın init'i olarak kullanılabilir.

In [ ]:
import sys, numpy as np
sys.path.insert(0, '/content/ev-drone-detector/src')
from ev_drone_detector.detection.detector import DroneDetector

detector = DroneDetector.from_config('/content/ev-drone-detector/configs/default.yaml')
detector.load_weights('/content/ev-drone-detector/checkpoints/best_iou.pt')

import glob
test_files = sorted(glob.glob('/content/ev-drone-detector/data/test/*.npz'))[:3]
for f in test_files:
    dets = detector.detect_from_npz(f)
    if dets:
        best = max(dets, key=lambda d: d['score'])
        print(f'{f.split("/")[-1]}: init_bbox={best["bbox"]}  score={best["score"]:.3f}')
    else:
        print(f'{f.split("/")[-1]}: (no detection)')